# Datapoint: Chip Component Spend

**What:** quarterly spend on four hardware-component categories — **Memory (HBM), Logic, Packaging (CoWoS), Auxiliary** — for AI accelerators designed by **NVIDIA, AMD, Google, Amazon**. Unit: **USD per quarter**.

**Targets:** **Q1 2026 nowcast** (Epoch actuals incomplete — missing NVIDIA) and a chained **Q2 2026 forecast**.

**Principle:** every model input traces to a real source. History is read from the source dataset; each external datasource is a structured JSON, and the model **indexes into those JSON figures** (no fabricated values). The few inputs no source quantifies are flagged `MODELING`.

Self-contained template: §1 History · §2 Datasources · §3 Model · §4 Forecast · §5 Visualization.

## 1. Datapoint History
Read from the actual source (Epoch AI `ai_chip_components`, CC-BY) and processed here — not hardcoded.

In [1]:
import io, json, urllib.request
from statistics import mean, pstdev
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

COMPONENTS = ["Memory", "Logic", "Packaging", "Auxiliary"]
COLORS = {
    "Memory": "#4fa8a0",
    "Logic": "#e0a44a",
    "Packaging": "#4a5fd0",
    "Auxiliary": "#d65a9a",
}
DESIGNERS = ["NVIDIA", "AMD", "Google", "Amazon"]
BASE_CSV = "../data/processed/epoch_ai/ai_chip_components__quarterly_by_chip.csv"
COST_COLUMNS = {
    "Memory": "HBM cost (USD) (median)",
    "Logic": "Logic cost (USD) (median)",
    "Packaging": "CoWoS cost (USD) (median)",
    "Auxiliary": "Auxiliary cost (USD) (median)",
}


def to_number(column):
    return (
        pl.col(column)
        .cast(pl.String)
        .str.replace_all(r"[$,%]", "")
        .cast(pl.Float64, strict=False)
    )


def quarter_sort_key(quarter):
    quarter_num, year = quarter.split()
    return int(year) * 4 + int(quarter_num[1])


chip_costs_raw = (
    pl.read_csv(
        BASE_CSV,
        infer_schema_length=20000,
        ignore_errors=True,
        truncate_ragged_lines=True,
    )
    .filter(pl.col("Designer").is_in(DESIGNERS))
    .with_columns(
        [
            to_number(column).alias(component)
            for component, column in COST_COLUMNS.items()
        ]
    )
)
quarterly_costs = (
    chip_costs_raw.group_by("Quarter")
    .agg(
        [
            (pl.col(component).sum() / 1e9).round(4).alias(component)
            for component in COMPONENTS
        ]
    )
    .with_columns(
        sort_key=pl.col("Quarter").map_elements(quarter_sort_key, return_dtype=pl.Int64)
    )
    .sort("sort_key")
)
# Q1 2026 is incomplete in Epoch (missing NVIDIA) -> nowcast it in section 4; history = complete quarters
history = (
    quarterly_costs.filter(pl.col("Quarter") != "Q1 2026")
    .drop("sort_key")
    .with_columns(Total=sum(pl.col(component) for component in COMPONENTS))
)
history

Quarter,Memory,Logic,Packaging,Auxiliary,Total
str,f64,f64,f64,f64,f64
"""Q1 2024""",1.6773,0.4579,0.6109,0.4875,3.2336
"""Q2 2024""",2.3065,0.5808,0.7745,0.6061,4.2679
"""Q3 2024""",3.3393,0.8227,1.073,0.7389,5.9739
"""Q4 2024""",4.8054,1.2289,1.5148,0.9523,8.5014
"""Q1 2025""",5.5342,1.4082,1.6949,1.2167,9.854
"""Q2 2025""",6.2846,1.5063,1.7979,1.1499,10.7387
"""Q3 2025""",8.7238,1.8867,2.2416,1.3816,14.2337
"""Q4 2025""",11.0007,2.2421,2.5715,1.5319,17.3462


In [2]:
history_long = history.unpivot(
    index="Quarter", on=COMPONENTS, variable_name="Component", value_name="spend"
)
figure = px.bar(
    history_long,
    x="Quarter",
    y="spend",
    color="Component",
    category_orders={"Component": COMPONENTS},
    color_discrete_map=COLORS,
    labels={"spend": "USD billions"},
    title="Chip component spend - history (NVIDIA/AMD/Google/Amazon)",
)
figure.update_layout(height=440)
figure

## 2. Datasources for datapoint
Each datasource is a structured JSON: provenance metadata + a `data` dict of **machine-readable figures transcribed from the source** (the model indexes these). FRED is fetched **live**.

In [3]:
FRED_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=IPG3344S"
try:
    request = urllib.request.Request(
        FRED_URL, headers={"User-Agent": "yuyan-research-bot/0.1 (diabhaque@gmail.com)"}
    )
    fred = pl.read_csv(
        io.StringIO(urllib.request.urlopen(request, timeout=60).read().decode())
    )
    fred_column = fred.columns[-1]
    fred = fred.with_columns(
        pl.col(fred_column).cast(pl.Float64, strict=False)
    ).drop_nulls(fred_column)
    FRED_YOY = round(fred[fred_column][-1] / fred[fred_column][-13] - 1, 4)
    FRED_YOY_PREV = round(
        fred[fred_column][-4] / fred[fred_column][-16] - 1, 4
    )  # one quarter earlier (momentum)
    FRED_AS_OF = str(fred[fred.columns[0]][-1])
except Exception as error:
    FRED_YOY, FRED_YOY_PREV, FRED_AS_OF = 0.099, 0.08, f"fallback ({error})"
print("FRED IPG3344S (live):", f"{FRED_YOY*100:+.1f}% YoY (latest {FRED_AS_OF})")


def make_datasource(
    provenance, name, component, role, targets, url, as_of, data, new_info=""
):
    return dict(
        provenance=provenance,
        name=name,
        component=component,
        role=role,
        targets=targets,
        new_info=new_info,
        url=url,
        as_of=as_of,
        data=data,
    )


COMPANY_PR, NEWS_COMPANY, NEWS_INDUSTRY, SEC_FILING, GOV_MACRO, ANALYST = (
    "Company press release",
    "External news - specific company",
    "External news - industry",
    "SEC filing",
    "Government / macro",
    "Analyst",
)
BOTH_TARGETS = ["Q1 2026", "Q2 2026"]

DATASOURCES = [
    make_datasource(
        COMPANY_PR,
        "TSMC",
        ["Logic", "Packaging"],
        "hard",
        ["Q1 2026"],
        "https://pr.tsmc.com/english/news/3294",
        "2026-04-10",
        {"q1_rev_busd": 35.9, "rev_yoy": 0.406},
    ),
    make_datasource(
        COMPANY_PR,
        "TSMC (Q2 guide)",
        ["Logic", "Packaging"],
        "hard",
        ["Q2 2026"],
        "https://pr.tsmc.com/english/news/3305",
        "2026-05-08",
        {
            "apr_rev_yoy": 0.175,
            "q1_rev_busd": 35.9,
            "q2_guide_low_busd": 39.0,
            "q2_guide_high_busd": 40.2,
        },
    ),
    make_datasource(
        COMPANY_PR,
        "SK hynix",
        ["Memory"],
        "hard",
        BOTH_TARGETS,
        "https://news.skhynix.com/q1-2026-business-results/",
        "2026-04-23",
        {"revenue_qoq": 0.60, "revenue_yoy": 1.98, "hbm_share": 0.57},
    ),
    make_datasource(
        COMPANY_PR,
        "Micron",
        ["Memory"],
        "hard",
        BOTH_TARGETS,
        "https://investors.micron.com/news-releases/news-release-details/micron-technology-inc-reports-results-second-quarter-fiscal-2026",
        "2026-03-18",
        {"revenue_yoy": 1.96},
    ),
    make_datasource(
        COMPANY_PR,
        "Samsung",
        ["Memory"],
        "qualitative",
        BOTH_TARGETS,
        "https://news.samsung.com/global/samsung-electronics-announces-first-quarter-2026-results",
        "2026-04-30",
        {"memory_op_profit_yoy_x": 50},
    ),
    make_datasource(
        COMPANY_PR,
        "NVIDIA",
        ["Logic", "Auxiliary"],
        "qualitative",
        BOTH_TARGETS,
        "https://nvidianews.nvidia.com/news/nvidia-announces-financial-results-for-first-quarter-fiscal-2027",
        "2026-05-20",
        {
            "total_rev_busd": 81.6,
            "total_yoy": 0.85,
            "dc_share": 0.90,
            "dc_yoy_approx": 1.0,
        },
    ),
    make_datasource(
        COMPANY_PR,
        "Astera Labs",
        ["Auxiliary"],
        "hard",
        BOTH_TARGETS,
        "https://ir.asteralabs.com/news-releases/news-release-details/astera-labs-reports-first-quarter-2026-financial-results",
        "2026-05-05",
        {"revenue_musd": 308.4, "revenue_qoq": 0.14, "revenue_yoy": 0.93},
        "connectivity",
    ),
    make_datasource(
        COMPANY_PR,
        "Monolithic Power",
        ["Auxiliary"],
        "hard",
        BOTH_TARGETS,
        "https://www.monolithicpower.com/en/company/investor-relations",
        "2026-04-30",
        {"ai_revenue_musd": 262.8, "ai_revenue_qoq": 0.126, "ai_revenue_yoy": 0.977},
        "power",
    ),
    make_datasource(
        COMPANY_PR,
        "Broadcom",
        ["Auxiliary"],
        "hard",
        BOTH_TARGETS,
        "https://www.prnewswire.com/news-releases/broadcom-inc-announces-second-quarter-fiscal-year-2026-financial-results-and-quarterly-dividend-302790698.html",
        "2026-06-05",
        {"ai_revenue_busd": 10.8, "ai_revenue_yoy": 1.43},
    ),
    make_datasource(
        NEWS_COMPANY,
        "TSMC HPC platform (earnings coverage)",
        ["Logic"],
        "hard",
        BOTH_TARGETS,
        "https://en.macromicro.me/blog/tsmc-q1-earnings-call-rare-capacity-expansion-as-the-ai-megatrend-takes-shape",
        "2026-04-16",
        {"hpc_share": 0.61, "hpc_qoq": 0.20, "prev_hpc_share": 0.55},
    ),
    make_datasource(
        NEWS_COMPANY,
        "NVIDIA Blackwell shipments (analyst)",
        ["Logic"],
        "analyst",
        BOTH_TARGETS,
        "https://finance.biggo.com/news/g-whcJ0Bq7sy_YQMfkDE",
        "2025-12-29",
        {"racks_2026": 60000, "gpus_2026": 4300000, "blackwell_share": 0.71},
        "volume",
    ),
    make_datasource(
        NEWS_COMPANY,
        "ASE (advanced packaging outlook)",
        ["Packaging"],
        "hard",
        BOTH_TARGETS,
        "https://www.alphapilot.tech/discover/ase-technology-forecasts-3-5-billion-advanced-packaging-revenue-by-2026-amid-ai-demand",
        "2026-01",
        {"adv_pkg_rev_busd_2026": 3.5, "adv_pkg_growth_yoy": 0.10},
        "independent-osat",
    ),
    make_datasource(
        NEWS_INDUSTRY,
        "TrendForce - DRAM/NAND contract price (Q1)",
        ["Memory"],
        "hard-price",
        ["Q1 2026"],
        "https://www.trendforce.com/presscenter/news/20260202-12911.html",
        "2026-02-02",
        {
            "conventional_dram_qoq_range": [0.90, 0.95],
            "server_dram_qoq": 0.90,
            "pc_dram_qoq": 1.00,
            "nand_qoq_range": [0.55, 0.60],
            "essd_qoq_range": [0.53, 0.58],
            "prior_dram_estimate_range": [0.55, 0.60],
        },
    ),
    make_datasource(
        NEWS_INDUSTRY,
        "TrendForce - DRAM/NAND contract price (Q2)",
        ["Memory"],
        "hard-price",
        ["Q2 2026"],
        "https://www.trendforce.com/presscenter/news/20260331-12995.html",
        "2026-03-31",
        {"dram_contract_qoq_range": [0.58, 0.63], "nand_qoq_range": [0.70, 0.75]},
    ),
    make_datasource(
        NEWS_INDUSTRY,
        "TrendForce - HBM bit supply / capex",
        ["Memory"],
        "hard",
        BOTH_TARGETS,
        "https://www.trendforce.com/presscenter/news/20251113-12780.html",
        "2025-11-13",
        {
            "bit_supply": "capacity-constrained",
            "micron_capex_busd": 13.5,
            "skhynix_capex_busd": 20.5,
        },
        "volume-ceiling",
    ),
    make_datasource(
        NEWS_INDUSTRY,
        "TrendForce - HBM bit demand & AI server shipments 2026",
        ["Memory", "Logic"],
        "hard",
        BOTH_TARGETS,
        "https://www.trendforce.com/presscenter/news/20251030-12762.html",
        "2025-10-30",
        {
            "hbm_bit_demand_yoy": 0.70,
            "hbm_bit_demand_yoy_2025": 1.30,
            "asic_hbm_demand_yoy": 0.80,
            "ai_server_shipments_yoy": 0.20,
            "gpu_shipments_yoy": 0.161,
            "asic_shipments_yoy": 0.446,
            "hbm_share_of_dram_wafers": 0.23,
        },
        "volume",
    ),
    make_datasource(
        NEWS_INDUSTRY,
        "TrendForce - CoWoS capacity",
        ["Packaging"],
        "hard",
        BOTH_TARGETS,
        "https://www.trendforce.com/news/2026/05/14/news-tsmc-sees-ai-wafer-demand-rising-11x-from-2022-2026-targets-cowos-with-24-hbm-stacks-in-2029/",
        "2026-05-14",
        {
            "capacity_end2025_wpm_range": [75, 80],
            "capacity_end2026_wpm_range": [115, 140],
        },
    ),
    make_datasource(
        NEWS_INDUSTRY,
        "Optical transceivers (InnoLight/Coherent)",
        ["Auxiliary"],
        "analyst",
        BOTH_TARGETS,
        "https://research.fpx.world/p/part-2-beyond-power-the-networking",
        "2026",
        {"market_2024_busd": 9, "market_2026_busd": 16, "innolight_yoy": 1.23},
        "optical",
    ),
    make_datasource(
        NEWS_INDUSTRY,
        "Murata MLCC (passives price)",
        ["Auxiliary"],
        "hard-price",
        ["Q2 2026"],
        "https://www.digitimes.com/news/a20260218VL202/murata-mlcc-capacity-ai-server-demand.html",
        "2026-03-17",
        {"price_increase_range": [0.15, 0.35], "mlcc_per_ai_server_x": [10, 15]},
        "passives-price",
    ),
    make_datasource(
        SEC_FILING,
        "Amkor (10-Q / 8-K)",
        ["Packaging"],
        "hard",
        BOTH_TARGETS,
        "https://www.sec.gov/Archives/edgar/data/0001047127/000104712726000017/amkr3312026erex-991.htm",
        "2026-04",
        {"q1_net_sales_musd": 1685, "capex_2026_busd_range": [2.5, 3.0]},
        "independent-osat",
    ),
    make_datasource(
        GOV_MACRO,
        "Korea MOTIE / customs",
        ["all"],
        "macro",
        BOTH_TARGETS,
        "https://en.sedaily.com/finance/2026/04/01/semiconductor-power-defies-war-monthly-exports-head-toward",
        "2026-04-01",
        {"semi_exports_yoy": 2.0},
    ),
    make_datasource(
        GOV_MACRO,
        "FRED IPG3344S (US semis IP)",
        ["all"],
        "macro",
        BOTH_TARGETS,
        "https://fred.stlouisfed.org/series/IPG3344S",
        FRED_AS_OF,
        {"ip_yoy": FRED_YOY, "ip_yoy_prev_q": FRED_YOY_PREV},
    ),
    make_datasource(
        ANALYST,
        "Morgan Stanley - VR200 NVL72 BoM",
        ["Memory", "Packaging", "Auxiliary"],
        "analyst",
        ["Q2 2026"],
        "https://www.tomshardware.com/tech-industry/artificial-intelligence/nvidias-memory-costs-soar-485-percent-latest-ai-systems-now-cost-usd7-8-million-to-build-memory-now-comprises-25-percent-of-the-total-cost-rubin-gpus-a-mere-usd50-000-apiece",
        "2026-06",
        {
            "memory_per_rack_growth_x": 4.35,
            "memory_share_vr200": 0.26,
            "pcb_growth_x": 2.33,
            "mlcc_growth_x": 1.82,
        },
        "memory-share-trajectory",
    ),
]
DATASOURCE_FIGURES = {
    datasource["name"]: datasource["data"] for datasource in DATASOURCES
}


def source_value(name, key):
    return DATASOURCE_FIGURES[name][key]


datasources_table = pl.DataFrame(
    [
        {
            "provenance": datasource["provenance"],
            "name": datasource["name"],
            "component": ",".join(datasource["component"]),
            "role": datasource["role"],
            "as_of": datasource["as_of"],
            "data": json.dumps(datasource["data"]),
        }
        for datasource in DATASOURCES
    ]
)
print(
    f"{len(DATASOURCES)} datasources across {datasources_table['provenance'].n_unique()} provenance channels"
)
with pl.Config(tbl_rows=30, fmt_str_lengths=70, tbl_width_chars=200):
    print(datasources_table.sort("provenance"))

FRED IPG3344S (live): +9.9% YoY (latest 2026-04-01)
23 datasources across 6 provenance channels
shape: (23, 6)
┌──────────────────────────────────┬─────────────────────────────────────────────────────┬────────────────────────────┬─────────────┬────────────┬─────────────────────────────────────────────────────┐
│ provenance                       ┆ name                                                ┆ component                  ┆ role        ┆ as_of      ┆ data                                                │
│ ---                              ┆ ---                                                 ┆ ---                        ┆ ---         ┆ ---        ┆ ---                                                 │
│ str                              ┆ str                                                 ┆ str                        ┆ str         ┆ str        ┆ str                                                 │
╞══════════════════════════════════╪═════════════════════════════════════════════════

## 3. Clear model for datapoint
**3a — Derive scalars from the datasource JSON.** Each per-component growth scalar is *indexed* from a source figure (or transparently derived from source numbers); inputs no source quantifies are flagged `MODELING`.

In [4]:
# ---- helpers ----
def quarterly_cagr(start, end, quarters):
    return (end / start) ** (1 / quarters) - 1


def range_midpoint(value_range):
    return sum(value_range) / len(value_range)


BAND = 0.30  # MODELING: relative half-width for low/high when the source gives a point (not a range)


def band(base, low=None, high=None):
    return {
        "low": round(low if low is not None else base * (1 - BAND), 4),
        "base": round(base, 4),
        "high": round(high if high is not None else base * (1 + BAND), 4),
    }


# ---- derived-from-source quantities ----
cowos_qoq = quarterly_cagr(
    range_midpoint(
        source_value("TrendForce - CoWoS capacity", "capacity_end2025_wpm_range")
    ),
    range_midpoint(
        source_value("TrendForce - CoWoS capacity", "capacity_end2026_wpm_range")
    ),
    4,
)  # SOURCED (derived)
optical_qoq = quarterly_cagr(
    source_value("Optical transceivers (InnoLight/Coherent)", "market_2024_busd"),
    source_value("Optical transceivers (InnoLight/Coherent)", "market_2026_busd"),
    8,
)  # SOURCED (derived)
nvidia_demand_qoq = (1 + source_value("NVIDIA", "dc_yoy_approx")) ** (
    1 / 4
) - 1  # SOURCED (NVIDIA DC ~2x YoY)
aux_supply_qoq = range_midpoint(
    [
        source_value("Astera Labs", "revenue_qoq"),
        source_value("Monolithic Power", "ai_revenue_qoq"),
    ]
)  # SOURCED
tsmc_q2_qoq = (
    range_midpoint(
        [
            source_value("TSMC (Q2 guide)", "q2_guide_low_busd"),
            source_value("TSMC (Q2 guide)", "q2_guide_high_busd"),
        ]
    )
    / source_value("TSMC (Q2 guide)", "q1_rev_busd")
    - 1
)  # SOURCED (derived)


# Memory VOLUME (HBM bit demand): SOURCED from TrendForce HBM bit-demand growth, converted YoY -> QoQ.
def qoq_from_yoy(yoy):
    return (1 + yoy) ** (1 / 4) - 1


hbm_demand_source = "TrendForce - HBM bit demand & AI server shipments 2026"
volume_qoq = {
    "low": round(
        qoq_from_yoy(source_value(hbm_demand_source, "ai_server_shipments_yoy")), 4
    ),  # AI-server units +20% YoY (content-flat floor)
    "base": round(
        qoq_from_yoy(source_value(hbm_demand_source, "hbm_bit_demand_yoy")), 4
    ),  # HBM bit demand +70% YoY
    "high": round(
        qoq_from_yoy(source_value(hbm_demand_source, "asic_hbm_demand_yoy")), 4
    ),
}  # ASIC HBM demand +80% YoY

# MACRO pace scalar: SOURCED from FRED semiconductor-IP momentum + Korea chip-export confirmation, bounded.
ip_acceleration = source_value("FRED IPG3344S (US semis IP)", "ip_yoy") - source_value(
    "FRED IPG3344S (US semis IP)", "ip_yoy_prev_q"
)
korea_exports_hot = source_value("Korea MOTIE / customs", "semi_exports_yoy") > 1.0
macro_tilt = max(
    -0.05, min(0.05, ip_acceleration + (0.02 if korea_exports_hot else -0.02))
)
macro_scaler = {
    "low": round(1 + macro_tilt - 0.03, 3),
    "base": round(1 + macro_tilt, 3),
    "high": round(1 + macro_tilt + 0.03, 3),
}

trendforce_q1, trendforce_q2 = (
    "TrendForce - DRAM/NAND contract price (Q1)",
    "TrendForce - DRAM/NAND contract price (Q2)",
)

TARGETS = {
    "Q1 2026": dict(
        kind="nowcast",
        # SOURCED: TrendForce Q1 DRAM. Proxy for HBM (HBM-specific QoQ not disclosed) -> documented.
        price_qoq={
            "low": source_value(trendforce_q1, "prior_dram_estimate_range")[0],
            "base": source_value(trendforce_q1, "server_dram_qoq"),
            "high": source_value(trendforce_q1, "conventional_dram_qoq_range")[1],
        },
        volume_qoq=volume_qoq,  # SOURCED HBM bit demand +70% YoY -> QoQ
        supply_qoq={
            "Memory": band(
                source_value("SK hynix", "revenue_qoq")
            ),  # SOURCED SK hynix +60% QoQ
            "Logic": band(
                source_value("TSMC HPC platform (earnings coverage)", "hpc_qoq")
            ),  # SOURCED TSMC HPC +20% QoQ
            "Packaging": band(cowos_qoq),  # SOURCED CoWoS capacity (derived)
            "Auxiliary": band(aux_supply_qoq),
        },  # SOURCED Astera/MPWR QoQ
        demand_qoq={
            component: band(nvidia_demand_qoq) for component in COMPONENTS
        },  # SOURCED NVIDIA DC ~2x YoY -> QoQ
        macro_scaler=macro_scaler,  # SOURCED FRED semi-IP momentum + Korea exports
        analyst_qoq=None,
        weights={
            "supply": 0.30,
            "price": 0.30,
            "trend": 0.20,
            "macro": 0.12,
            "demand": 0.08,
        },
    ),
    "Q2 2026": dict(
        kind="forecast",
        price_qoq={
            "low": source_value(trendforce_q2, "dram_contract_qoq_range")[
                0
            ],  # SOURCED TrendForce Q2 DRAM +58-63%
            "base": range_midpoint(
                source_value(trendforce_q2, "dram_contract_qoq_range")
            ),
            "high": source_value(trendforce_q2, "dram_contract_qoq_range")[1],
        },
        volume_qoq=volume_qoq,  # SOURCED HBM bit demand +70% YoY -> QoQ
        supply_qoq={
            "Memory": band(
                range_midpoint(source_value(trendforce_q2, "dram_contract_qoq_range"))
            ),  # SOURCED memory price->rev proxy (Q2)
            "Logic": band(
                tsmc_q2_qoq,  # SOURCED TSMC Q2 guide
                source_value("TSMC (Q2 guide)", "q2_guide_low_busd")
                / source_value("TSMC (Q2 guide)", "q1_rev_busd")
                - 1,
                source_value("TSMC (Q2 guide)", "q2_guide_high_busd")
                / source_value("TSMC (Q2 guide)", "q1_rev_busd")
                - 1,
            ),
            "Packaging": band(cowos_qoq),  # SOURCED CoWoS (derived)
            "Auxiliary": band(aux_supply_qoq),
        },  # SOURCED Astera/MPWR (+MLCC price below)
        demand_qoq={
            component: band(nvidia_demand_qoq) for component in COMPONENTS
        },  # SOURCED
        macro_scaler=macro_scaler,  # SOURCED FRED semi-IP momentum + Korea exports
        # MODELING: MS BoM is a gen-over-gen / H2 trajectory, not a Q2 QoQ; used as a small directional tilt.
        analyst_qoq={
            "Memory": {"low": 0.25, "base": 0.40, "high": 0.60},
            "Logic": {"low": 0.05, "base": 0.10, "high": 0.16},
            "Packaging": {"low": 0.06, "base": 0.12, "high": 0.20},
            "Auxiliary": {"low": 0.05, "base": 0.10, "high": 0.18},
        },
        weights={
            "supply": 0.26,
            "price": 0.26,
            "trend": 0.18,
            "analyst": 0.12,
            "macro": 0.10,
            "demand": 0.08,
        },
    ),
}
print("Derived base scalars (sourced):")
print(
    f"  CoWoS QoQ (Packaging) = {cowos_qoq:.3f}; TSMC Q2 QoQ (Logic) = {tsmc_q2_qoq:.3f}; "
    f"NVIDIA demand QoQ = {nvidia_demand_qoq:.3f}; Aux supply QoQ = {aux_supply_qoq:.3f}"
)
print(
    f"  Memory price Q1 base = {TARGETS['Q1 2026']['price_qoq']['base']} (TrendForce server DRAM); "
    f"Q2 base = {TARGETS['Q2 2026']['price_qoq']['base']:.3f} (TrendForce Q2 DRAM range)"
)
print(
    f"  Memory volume_qoq = {volume_qoq} (HBM bit demand +70% YoY -> QoQ; floor=AI-server units, high=ASIC-HBM)"
)
print(
    f"  macro_scaler = {macro_scaler} (FRED semi-IP momentum {ip_acceleration:+.3f} + Korea exports confirmation)"
)

Derived base scalars (sourced):
  CoWoS QoQ (Packaging) = 0.133; TSMC Q2 QoQ (Logic) = 0.103; NVIDIA demand QoQ = 0.189; Aux supply QoQ = 0.133
  Memory price Q1 base = 0.9 (TrendForce server DRAM); Q2 base = 0.605 (TrendForce Q2 DRAM range)
  Memory volume_qoq = {'low': 0.0466, 'base': 0.1419, 'high': 0.1583} (HBM bit demand +70% YoY -> QoQ; floor=AI-server units, high=ASIC-HBM)
  macro_scaler = {'low': 0.969, 'base': 0.999, 'high': 1.029} (FRED semi-IP momentum -0.021 + Korea exports confirmation)


**3b — The engine.** Per component: build estimate families and reconcile (suppliers≡buyers, no double-count); Memory price×volume decomposition; compute confidence from `n_anchors` + dispersion, and a data-driven band. Q2 chains off Q1 (uncertainty added in quadrature).

In [5]:
ANCHOR_ROLES = {"hard", "hard-price", "analyst"}
BASE_HW, CV_REF = 0.12, 0.35
CONF_HIGH = dict(min_anchors=2, max_cv=0.20)
CONF_MED = dict(min_anchors=1, max_cv=0.30)


def median_qoq(series):
    quarter_growths = [series[i] / series[i - 1] - 1 for i in range(1, len(series))]
    return sorted(quarter_growths)[len(quarter_growths) // 2]


def count_anchors(component, target):
    return len(
        {
            datasource["name"]
            for datasource in DATASOURCES
            if target in datasource["targets"]
            and component in datasource["component"]
            and datasource["role"] in ANCHOR_ROLES
        }
    )


def build_estimates(component, level, median_growth, scenario, target):
    family_estimates = {
        "trend": level * (1 + median_growth),
        "supply": level * (1 + target["supply_qoq"][component][scenario]),
        "demand": level * (1 + target["demand_qoq"][component][scenario]),
        "macro": level * (1 + median_growth) * target["macro_scaler"][scenario],
    }
    family_estimates["price"] = (
        level
        * (1 + target["volume_qoq"][scenario])
        * (1 + target["price_qoq"][scenario])
        if component == "Memory"
        else family_estimates["trend"]
    )
    if target.get("analyst_qoq"):
        family_estimates["analyst"] = level * (
            1 + target["analyst_qoq"][component][scenario]
        )
    family_estimates["reconciled"] = sum(
        target["weights"][family] * family_estimates[family]
        for family in target["weights"]
    )
    return family_estimates


def classify_confidence(anchor_count, dispersion_cv):
    if anchor_count >= CONF_HIGH["min_anchors"] and dispersion_cv < CONF_HIGH["max_cv"]:
        label = "high"
    elif anchor_count >= CONF_MED["min_anchors"] and dispersion_cv < CONF_MED["max_cv"]:
        label = "medium"
    else:
        label = "low"
    return label, round(
        min(1.0, anchor_count / 3) * max(0.0, 1 - dispersion_cv / CV_REF), 2
    )


def run_target(target_name, base_levels, relative_prior=0.0):
    target = TARGETS[target_name]
    rows, detail_rows = [], []
    for component in COMPONENTS:
        median_growth = median_qoq(history[component].to_list())
        level = base_levels[component]
        base_est = build_estimates(component, level, median_growth, "base", target)
        low_est = build_estimates(component, level, median_growth, "low", target)
        high_est = build_estimates(component, level, median_growth, "high", target)
        reconciled_base = base_est["reconciled"]
        families = [family for family in base_est if family != "reconciled"]
        spread = [
            estimate[family]
            for estimate in (low_est, base_est, high_est)
            for family in families
        ]
        dispersion_cv = pstdev(spread) / mean(spread) if mean(spread) else 0.0
        anchor_count = count_anchors(component, target_name)
        half_width = (
            dispersion_cv**2 + (BASE_HW / (anchor_count + 1) ** 0.5) ** 2
        ) ** 0.5
        if relative_prior:
            half_width = (half_width**2 + relative_prior**2) ** 0.5
        label, score = classify_confidence(anchor_count, dispersion_cv)
        rows.append(
            dict(
                component=component,
                reconciled_base=round(reconciled_base, 3),
                low=round(reconciled_base * (1 - half_width), 3),
                high=round(reconciled_base * (1 + half_width), 3),
                trend_only=round(base_est["trend"], 3),
                confidence=label,
                confidence_score=score,
                n_anchors=anchor_count,
                dispersion_cv=round(dispersion_cv, 3),
            )
        )
        for family in [
            name
            for name in [
                "trend",
                "supply",
                "demand",
                "price",
                "macro",
                "analyst",
                "reconciled",
            ]
            if name in base_est
        ]:
            detail_rows.append(
                dict(
                    component=component,
                    estimate=family,
                    value_b=round(base_est[family], 3),
                )
            )
    result = pl.DataFrame(rows)
    total_row = {"component": "TOTAL", "confidence": "-"}
    for column in ["reconciled_base", "low", "high", "trend_only"]:
        total_row[column] = round(result[column].sum(), 3)
    return pl.concat(
        [result, pl.DataFrame([total_row])], how="diagonal_relaxed"
    ), pl.DataFrame(detail_rows)

## 4. Datapoint Forecast

In [6]:
q4_2025_levels = {
    component: history.filter(pl.col("Quarter") == "Q4 2025")[component][0]
    for component in COMPONENTS
}
q1_result, q1_detail = run_target("Q1 2026", q4_2025_levels)
q1_total = q1_result.filter(pl.col("component") == "TOTAL").row(0, named=True)
q1_bases = {
    row["component"]: row["reconciled_base"]
    for row in q1_result.filter(pl.col("component") != "TOTAL").iter_rows(named=True)
}
q1_relative_uncertainty = ((q1_total["high"] - q1_total["low"]) / 2) / q1_total[
    "reconciled_base"
]
q2_result, q2_detail = run_target("Q2 2026", q1_bases, q1_relative_uncertainty)
q2_total = q2_result.filter(pl.col("component") == "TOTAL").row(0, named=True)
print(
    f"Q1 2026 nowcast  total: ${q1_total['reconciled_base']:.1f}B  (low ${q1_total['low']:.1f} - high ${q1_total['high']:.1f}B)"
)
print(
    f"Q2 2026 forecast total: ${q2_total['reconciled_base']:.1f}B  (low ${q2_total['low']:.1f} - high ${q2_total['high']:.1f}B)  (+{(q2_total['reconciled_base']/q1_total['reconciled_base']-1)*100:.0f}% QoQ)"
)
q1_result

Q1 2026 nowcast  total: $26.0B  (low $21.5 - high $30.5B)
Q2 2026 forecast total: $37.4B  (low $29.2 - high $45.7B)  (+44% QoQ)


component,reconciled_base,low,high,trend_only,confidence,confidence_score,n_anchors,dispersion_cv
str,f64,f64,f64,f64,str,f64,i64,f64
"""Memory""",18.326,14.355,22.297,15.127,"""medium""",0.4,5,0.211
"""Logic""",2.761,2.584,2.939,2.808,"""high""",0.9,4,0.036
"""Packaging""",3.106,2.889,3.322,3.206,"""high""",0.87,4,0.044
"""Auxiliary""",1.824,1.705,1.943,1.868,"""high""",0.89,4,0.037
"""TOTAL""",26.017,21.533,30.501,23.009,"""-""",null,null,null


In [7]:
q2_result

component,reconciled_base,low,high,trend_only,confidence,confidence_score,n_anchors,dispersion_cv
str,f64,f64,f64,f64,str,f64,i64,f64
"""Memory""",28.256,21.725,34.787,25.201,"""high""",0.58,6,0.147
"""Logic""",3.286,2.66,3.912,3.458,"""high""",0.83,4,0.061
"""Packaging""",3.718,3.024,4.413,3.873,"""high""",0.85,5,0.053
"""Auxiliary""",2.152,1.755,2.549,2.224,"""high""",0.87,6,0.047
"""TOTAL""",37.412,29.164,45.661,34.756,"""-""",null,null,null


## 5. Appropriate Visualization

In [8]:
figure = go.Figure()
figure.add_scatter(
    x=history["Quarter"].to_list(),
    y=history["Total"].to_list(),
    mode="lines+markers",
    name="history (actual)",
)
for label, total, color in [
    ("Q1 2026 nowcast", q1_total, "crimson"),
    ("Q2 2026 forecast", q2_total, "darkorange"),
]:
    figure.add_scatter(
        x=[label[:7]],
        y=[total["reconciled_base"]],
        mode="markers",
        name=label,
        marker=dict(size=11, color=color),
        error_y=dict(
            type="data",
            symmetric=False,
            array=[total["high"] - total["reconciled_base"]],
            arrayminus=[total["reconciled_base"] - total["low"]],
        ),
    )
figure.update_layout(
    title="Chip component spend: history + Q1 nowcast + Q2 forecast",
    yaxis_title="USD billions",
    height=480,
)
figure

In [9]:
history_long = history.unpivot(
    index="Quarter", on=COMPONENTS, variable_name="Component", value_name="spend"
)


def reconciled_rows(result, quarter_name):
    return result.filter(pl.col("component") != "TOTAL").select(
        Quarter=pl.lit(quarter_name),
        Component=pl.col("component"),
        spend=pl.col("reconciled_base"),
    )


breakdown_bars = pl.concat(
    [
        history_long.select(["Quarter", "Component", "spend"]),
        reconciled_rows(q1_result, "Q1 2026"),
        reconciled_rows(q2_result, "Q2 2026"),
    ],
    how="vertical_relaxed",
)
quarter_order = history["Quarter"].to_list() + ["Q1 2026", "Q2 2026"]
figure = px.bar(
    breakdown_bars,
    x="Quarter",
    y="spend",
    color="Component",
    category_orders={"Quarter": quarter_order, "Component": COMPONENTS},
    color_discrete_map=COLORS,
    labels={"spend": "USD billions"},
    title="Component breakdown: history -> Q1 nowcast -> Q2 forecast",
)
figure.add_annotation(
    x="Q1 2026",
    y=q1_total["reconciled_base"],
    text="nowcast",
    showarrow=True,
    arrowhead=2,
    yshift=8,
)
figure.add_annotation(
    x="Q2 2026",
    y=q2_total["reconciled_base"],
    text="forecast",
    showarrow=True,
    arrowhead=2,
    yshift=8,
)
figure.update_layout(height=540)
figure

In [10]:
family_order = ["trend", "supply", "demand", "price", "macro", "analyst", "reconciled"]
figure = px.bar(
    q2_detail.to_pandas(),
    x="estimate",
    y="value_b",
    facet_col="component",
    color="estimate",
    category_orders={"estimate": family_order, "component": COMPONENTS},
    labels={"value_b": "USD billions"},
    title="Q2 2026: estimate by family, per component",
)
figure.update_yaxes(matches=None)
figure.update_layout(height=430, showlegend=False)
figure

In [11]:
confidence_table = pl.concat(
    [
        q1_result.filter(pl.col("component") != "TOTAL")
        .select(["component", "confidence", "confidence_score", "n_anchors"])
        .with_columns(target=pl.lit("Q1 2026")),
        q2_result.filter(pl.col("component") != "TOTAL")
        .select(["component", "confidence", "confidence_score", "n_anchors"])
        .with_columns(target=pl.lit("Q2 2026")),
    ]
)
figure = px.bar(
    confidence_table.to_pandas(),
    x="component",
    y="confidence_score",
    color="target",
    barmode="group",
    text="confidence",
    category_orders={"component": COMPONENTS},
    labels={"confidence_score": "confidence (0-1)"},
    title="Computed confidence by component",
)
figure.update_traces(textposition="outside")
figure.update_layout(height=420)
figure

## Summary & limitations
- Forecast totals and per-component bases are in §4 (each model scalar is **indexed from the datasource JSON** in §3a; provenance printed there).
- **Sourced** scalars: Memory price (TrendForce DRAM), Memory **volume** (TrendForce HBM bit demand +70% YoY), Memory supply (SK hynix), Logic (TSMC HPC / Q2 guide), Packaging (CoWoS capacity), Auxiliary (Astera/MPWR), demand (NVIDIA DC), **macro** (FRED semi-IP momentum + Korea exports). **`MODELING`** (no source figure): only the Q2 analyst tilt (MS BoM is gen-over-gen) and the low/high `BAND` around point estimates.
- **Memory price caveat:** TrendForce quantifies DRAM/server-DRAM, not HBM specifically; we index server DRAM as a documented proxy (likely an upper bound, since HBM is LTA-priced).
- All inputs are real and traceable (URL + as_of in §2). **This is a forecast, not a guarantee.**